# 05 · Pronóstico de demanda y alerta de saturación
Saturación = ocupación > P90 histórico. Baselines estadísticos primero, luego ML.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
occ = pd.read_csv(config.DATA_PROCESSED / 'ocupacion_horaria.csv', parse_dates=['fecha_hora']).set_index('fecha_hora')['ocupacion']
umbral = occ.quantile(.9)
sat = (occ > umbral).astype(int)
print(f'Umbral: {umbral:.0f} pacientes · {sat.mean():.1%} de horas saturadas')

## Split temporal y baseline estacional (misma hora, semana anterior)

In [ ]:
cut = occ.index[int(len(occ)*.8)]
train, test = occ[:cut], occ[cut:]
naive = occ.shift(24*7)[cut:]
from sklearn.metrics import mean_absolute_error, roc_auc_score
print('MAE naive semanal:', round(mean_absolute_error(test, naive.fillna(train.mean())), 2))

## Holt-Winters (estacionalidad diaria)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
hw = ExponentialSmoothing(train, seasonal='add', seasonal_periods=24, trend=None).fit()
hw_pred = hw.forecast(len(test))
print('MAE Holt-Winters:', round(mean_absolute_error(test, hw_pred), 2))

## LightGBM con rezagos y calendario → probabilidad de saturación a h horas

In [ ]:
import lightgbm as lgb
H = 6  # horizonte en horas
d = pd.DataFrame({'y': occ})
for l in [H, H+1, H+2, 24, 48, 168]:
    d[f'lag_{l}'] = occ.shift(l)
d['roll24'] = occ.shift(H).rolling(24).mean()
d['hora'], d['dia'], d['mes'] = d.index.hour, d.index.dayofweek, d.index.month
d['target_sat'] = (d['y'] > umbral).astype(int)
d = d.dropna()
Xf = d.drop(columns=['y','target_sat'])
tr, te = d.index < cut, d.index >= cut
m = lgb.LGBMClassifier(n_estimators=400, learning_rate=.05, verbose=-1).fit(Xf[tr], d['target_sat'][tr])
p = m.predict_proba(Xf[te])[:,1]
print(f'AUC saturación a {H} h:', round(roc_auc_score(d['target_sat'][te], p), 3))

In [ ]:
ax = test.plot(figsize=(12,3), label='Real', alpha=.7)
hw_pred.plot(ax=ax, label='Holt-Winters')
ax.axhline(umbral, c='r', ls='--', label='Umbral')
ax.legend(); ax.set_xlim(test.index[0], test.index[0]+pd.Timedelta(days=14));

## Guardar para Power BI

In [ ]:
pd.DataFrame({'fecha_hora': test.index, 'real': test.values, 'pronostico_hw': hw_pred.values}).to_csv(config.DATA_PROCESSED / 'pronostico_ocupacion.csv', index=False)
pd.DataFrame({'fecha_hora': Xf[te].index, 'prob_saturacion': p}).to_csv(config.DATA_PROCESSED / 'alerta_saturacion.csv', index=False)